# ViTPose 포즈 추출 실습 (Colab)
사진을 업로드하면 **사람 검출(RT-DETR) → 포즈 추정(ViTPose)** 을 수행하고
- 스켈레톤이 그려진 결과 이미지
- 사람별 **detection confidence**, 17개 키포인트별 **confidence(score)** 표

를 출력합니다. HF 데모(hysts/ViTPose-transformers)와 동일한 모델을 사용합니다.

> 런타임 → 런타임 유형 변경 → **GPU(T4)** 권장 (CPU도 동작, 이미지당 수 초 소요)

In [ ]:
#@title 1. 라이브러리 설치 (최초 1회, 약 1분)
!pip -q install transformers==4.49.0 supervision==0.25.0
# pandas/pillow는 Colab 기본 버전 사용 (업그레이드하면 google-colab과 충돌)

In [ ]:
#@title 2. 모델 로드
import torch, numpy as np, pandas as pd
from PIL import Image
from transformers import AutoProcessor, RTDetrForObjectDetection, VitPoseForPoseEstimation

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

det_processor = AutoProcessor.from_pretrained("PekingU/rtdetr_r50vd_coco_o365")
det_model = RTDetrForObjectDetection.from_pretrained("PekingU/rtdetr_r50vd_coco_o365").to(device).eval()

POSE_CKPT = "usyd-community/vitpose-base-simple"  # 더 정확한 모델: "usyd-community/vitpose-plus-base"
pose_processor = AutoProcessor.from_pretrained(POSE_CKPT)
pose_model = VitPoseForPoseEstimation.from_pretrained(POSE_CKPT).to(device).eval()

COCO_NAMES = ["nose","left_eye","right_eye","left_ear","right_ear","left_shoulder","right_shoulder",
              "left_elbow","right_elbow","left_wrist","right_wrist","left_hip","right_hip",
              "left_knee","right_knee","left_ankle","right_ankle"]
print("모델 로드 완료")

In [ ]:
#@title 3. 추론 함수 정의
import supervision as sv

DET_THRESHOLD = 0.2   #@param {type:"number"}
KPT_THRESHOLD = 0.3   #@param {type:"number"}

@torch.no_grad()
def detect_persons(image):
    inputs = det_processor(images=image, return_tensors="pt").to(device)
    outputs = det_model(**inputs)
    results = det_processor.post_process_object_detection(
        outputs, target_sizes=torch.tensor([(image.height, image.width)]), threshold=DET_THRESHOLD)[0]
    keep = results["labels"] == 0  # COCO person
    boxes = results["boxes"][keep].cpu().numpy()   # xyxy
    scores = results["scores"][keep].cpu().numpy()
    return boxes, scores

@torch.no_grad()
def estimate_pose(image, boxes_xyxy):
    if len(boxes_xyxy) == 0:
        return np.zeros((0,17,2)), np.zeros((0,17))
    boxes_xywh = boxes_xyxy.copy()
    boxes_xywh[:, 2] -= boxes_xywh[:, 0]; boxes_xywh[:, 3] -= boxes_xywh[:, 1]
    inputs = pose_processor(image, boxes=[boxes_xywh], return_tensors="pt").to(device)
    if "plus" in POSE_CKPT:
        inputs["dataset_index"] = torch.tensor([0]*len(boxes_xywh), device=device)
    outputs = pose_model(**inputs)
    res = pose_processor.post_process_pose_estimation(outputs, boxes=[boxes_xywh])[0]
    kps = np.stack([r["keypoints"].cpu().numpy() for r in res])
    scs = np.stack([r["scores"].cpu().numpy() for r in res])
    return kps, scs

def run(image: Image.Image):
    image = image.convert("RGB")
    boxes, det_scores = detect_persons(image)
    kps, kp_scores = estimate_pose(image, boxes)

    # ---- 시각화 ----
    frame = np.array(image)
    if len(boxes):
        dets = sv.Detections(xyxy=boxes, confidence=det_scores, class_id=np.zeros(len(boxes), int))
        frame = sv.BoxAnnotator(thickness=2).annotate(frame, dets)
        frame = sv.LabelAnnotator().annotate(frame, dets, labels=[f"person {i} {s:.2f}" for i, s in enumerate(det_scores)])
        #kp_obj = sv.KeyPoints(xy=kps.astype(np.float32), confidence=kp_scores.astype(np.float32))
        kp_obj = sv.KeyPoints(xy=kps.astype(np.float32))
        frame = sv.EdgeAnnotator(color=sv.Color.GREEN, thickness=2).annotate(frame, kp_obj)
        frame = sv.VertexAnnotator(color=sv.Color.RED, radius=4).annotate(frame, kp_obj)

    # ---- 점수 표 ----
    rows = []
    for i, (ds, sc) in enumerate(zip(det_scores, kp_scores)):
        row = {"person": i, "det_conf": round(float(ds), 3),
               "kpt_mean": round(float(sc.mean()), 3), "kpt_min": round(float(sc.min()), 3),
               f"n_kpt>={KPT_THRESHOLD}": int((sc >= KPT_THRESHOLD).sum())}
        row.update({n: round(float(s), 3) for n, s in zip(COCO_NAMES, sc)})
        rows.append(row)
    df = pd.DataFrame(rows)
    return Image.fromarray(frame), df

print("준비 완료")

In [ ]:
#@title 4. 사진 업로드 → 결과 출력
from google.colab import files
from IPython.display import display

uploaded = files.upload()
for fname in uploaded:
    img = Image.open(fname)
    vis, df = run(img)
    print(f"\n===== {fname} : 검출된 사람 {len(df)}명 =====")
    display(vis.resize((vis.width * 640 // max(vis.width, vis.height), vis.height * 640 // max(vis.width, vis.height))))
    if len(df) == 0:
        print("사람이 검출되지 않았습니다. (DET_THRESHOLD를 낮춰보세요)")
        continue
    display(df[["person", "det_conf", "kpt_mean", "kpt_min", f"n_kpt>={KPT_THRESHOLD}"]])
    print("\n키포인트별 confidence:")
    display(df.set_index("person")[COCO_NAMES].T.style.background_gradient(cmap="RdYlGn", vmin=0, vmax=1))
    vis.save(f"result_{fname}")
    df.to_csv(f"scores_{fname.rsplit('.',1)[0]}.csv", index=False)

### 출력 설명
| 열 | 의미 |
|---|---|
| `det_conf` | RT-DETR이 "사람"이라고 판단한 확신도 (0~1) |
| `kpt_mean` / `kpt_min` | 17개 키포인트 confidence의 평균 / 최솟값 |
| `n_kpt>=0.3` | 임계값 이상으로 검출된 키포인트 개수 (17점 만점) |
| 키포인트 표 | 관절별 confidence (초록=확실, 빨강=불확실) |

결과 이미지(`result_*.jpg`)와 점수 CSV(`scores_*.csv`)는 왼쪽 파일 탭에서 다운로드할 수 있습니다.

### (선택) 웹캠으로 바로 찍기
아래 셀을 실행하면 Colab에서 웹캠 촬영 후 바로 분석합니다.